# constructing 2D/3D local box using GeoPoints.jl 

Nobuaki Fuji (IPGP/UPC/IUF) December 2025

todo: normals computations for each point, SeisTomoPy 


you can also see how to use lazyProduceOrLoad, which is not safe solution but when debugging, it's cool! (but don't write a biiiig jdl2 neither!)

In [ ]:
# Locate flexOPT securely without relying on @__DIR__ (unreliable in IJulia).
# If this notebook is outside the repository, set ENV["FLEXOPT_ROOT"] first.
import Pkg

function find_flexopt_root(start_dir=pwd())
    candidates = String[]
    if haskey(ENV, "FLEXOPT_ROOT")
        push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    end
    directory = abspath(start_dir)
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    for candidate in unique(candidates)
        project_file = joinpath(candidate, "Project.toml")
        source_dir = joinpath(candidate, "src")
        if isfile(project_file) && isfile(joinpath(source_dir, "commonBatchs.jl"))
            return candidate
        end
    end
    error("Cannot locate flexOPT. Start Jupyter inside the repository or set ENV[\"FLEXOPT_ROOT\"] to its absolute path.")
end

flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
@show VERSION Threads.nthreads() Base.active_project()
\ninclude(joinpath(flexopt_root, "src", "commonBatchs.jl"))
include(joinpath(flexopt_root, "src", "planet1D.jl"))
planet1D.configure_input!()
include(joinpath(flexopt_root, "src", "GeoPoints.jl"))

using .commonBatchs, .planet1D, .GeoPoints
using Colors\n

In [ ]:
set_default_planet!(:Earth) # if you wish to go to other telluric bodies, otherwise this is not necessary

## now p0 option is also available (17/06/2026)

In [ ]:
p1 = GeoPoint(43.6047,1.4447) # Toulouse (latitude, longitude)
p2 = GeoPoint(42.8485,1.6048) # Tarascon (à peu près)


In [ ]:
Δx = 500.0 # in metre
Δy = 500.0
Δz = 500.0

altMax = 10.e3 # in metre
altMin = -100.e3 # in metre

horizontalDepth = 50.e3

In [ ]:
set_default_planet!(:Earth) 
# two (extreme) points that can define the slice (or the x-y local plane for 3D box)
p1 = GeoPoint(48.8566,2.3522) # Paris (latitude, longitude)
p2 = GeoPoint(42.8,1.5) # Tarascon (à peu près)


Δx = 100.0 # in metre
Δz = 100.0

altMax = 10.e3 # in metre
altMin = -100.e3 # in metre

# altMax and altMin are measured from the middle point at p1 and p2 (but the planet's surface normally), the user can change the option hidden in constructLocalBox

In [ ]:
@show p1 # GeoPoint has some attributes

In [ ]:
# make a box/rectangle

boxGrids=constructLocalBox(p1,p2,Δx,Δz,altMin,altMax)
#allGridsInGeoPoints, allGridsInCartesian, effectiveRadii=lazyProduceOrLoad("boxGrids",constructLocalBox,p1,p2,Δx,Δz,altMin,altMax) <- don't do this, it's too heavy


In [ ]:
boxGrids.allGridsInGeoPoints[1,1,1]

In [ ]:
boxGrids.allGridsInCartesian[20,30]

In [ ]:
#seismicModel=getParamsAndTopo(allGridsInGeoPoints,effectiveRadii,2.0) # this can be also GPUed
seismicModel=lazyProduceOrLoad("seismicModel",getParamsAndTopo,boxGrids.allGridsInGeoPoints,boxGrids.effectiveRadii,2.0)
#seismicModel=lazyProduceOrLoad("seismicModel") # this is the laziest way to load

In [ ]:
Nx,Nz=size(seismicModel.ρ)

In [ ]:
maximum(seismicModel.ρ)

In [ ]:
using CairoMakie
Nx,Nz=size(seismicModel.ρ)
xvals = [p.xz[1] for p in boxGrids.allGridsInCartesian[:,1]]*1.e-3
zvals = [p.xz[2] for p in boxGrids.allGridsInCartesian[1,:]]*1.e-3
fig, ax, hm = heatmap(
    #topo.x,topo.y,topo.z';
    #collect((0:1:(Nx-1)).*Δx).*1.e-3,(collect(0:1:(Nz-1)).*Δz.+altMin).*1.e-3, seismicModel.ρ;
    xvals, zvals, seismicModel.ρ;
    colormap = :seismic,
    #colorrange=(0,4),
    axis = (aspect = DataAspect(), xlabel = "horizontal", ylabel = "depth from p1", title = "density model")
)
Colorbar(fig[1,2], hm, label="density")
fig

# if you want to put some 2D/3D perturbation into it (in percent or absolute value, as you wish)


In [ ]:
imageFilePer="./myPerturbation.png"
modelPer= read2DimageModel(imageFilePer; Ncolor=256, colorbar = [RGB(1.0,0.0,0.0), RGB(1.0,1.0,1.0), RGB(0.0,0.0,1.0)] ,values = [-1.0,0.0,1.0],reverseOrNot=true,showRecoveredImage=false)
seismicModel=lazyProduceOrLoad("seismicModel")
# here is the size of the region to be perturbed
nx,nz=size(seismicModel.Vpv)
nx1,nz1 = round(Int,nx*0.1),round(Int,nz*0.1)
nx2,nz2 = round(Int,nx*0.9),round(Int,nz*0.9)
VpvSub = seismicModel.Vpv[nx1:nx2,nz1:nz2]
newPer = adjustArray(VpvSub,modelPer)
@. VpvSub = (1.0+0.2*newPer)*VpvSub # a very exaggerated model
seismicModel.Vpv[nx1:nx2,nz1:nz2]= VpvSub[1:end,1:end];

In [ ]:
using CairoMakie
xvals = [p.xz[1] for p in boxGrids.allGridsInCartesian[:,1]]*1.e-3
zvals = [p.xz[2] for p in boxGrids.allGridsInCartesian[1,:]]*1.e-3
fig, ax, hm = heatmap(
    #topo.x,topo.y,topo.z';
    #collect((0:1:(Nx-1)).*Δx).*1.e-3,(collect(0:1:(Nz-1)).*Δz.+altMin).*1.e-3, seismicModel.ρ;
    xvals, zvals, seismicModel.Vpv;
    colormap = :seismic,
    #colorrange=(0,4),
    axis = (aspect = DataAspect(), xlabel = "horizontal", ylabel = "depth from p1", title = "Vpv model")
)
Colorbar(fig[1,2], hm, label="Vpv")
fig

# 3D box ? yes

In [ ]:
p1 = GeoPoint(35.4139,138.2665) # 
p2 = GeoPoint(35.4172,139.2596) # 



Δx = 500.0 # in metre
Δy = 500.0
Δz = 500.0

altMax = 10.e3 # in metre
altMin = -300.e3 # in metre

horizontalDepth = 50.e3

# with theparameters below we can 'see' the topo 
#p1 = GeoPoint(35.538067,138.673722)
#p2 = GeoPoint(35.255127,138.675729)

#Δx = 100.0 # in metre
#Δy = 100.0
#Δz = 100.0

#altMax = 5.e3 # in metre
#altMin = -2.e3 # in metre

#horizontalDepth = 20.e3

In [ ]:

boxGrids3D=constructLocalBox(p1,p2,Δx,Δy,Δz,-horizontalDepth,horizontalDepth,altMin,altMax)

#ok 3D box version needs to be GPUed

In [ ]:
#seismicModel3D=getParamsAndTopo(allGridsInGeoPoints3D,effectiveRadii3D,2.0)
#seismicModel3D=lazyProduceOrLoad("seismicModel3D_Fuji_Floriane_topo_enhanced",getParamsAndTopo,boxGrids3D.allGridsInGeoPoints,boxGrids3D.effectiveRadii,2.0)
seismicModel3D=lazyProduceOrLoad("seismicModel3D_Fuji_Floriane",getParamsAndTopo,boxGrids3D.allGridsInGeoPoints,boxGrids3D.effectiveRadii,2.0)
#seismicModel3D=lazyProduceOrLoad("seismicModel3D")

In [ ]:
using GLMakie
GLMakie.activate!()
Makie.inline!() 
Nx3D,Ny3D,Nz3D=boxGrids3D.Nx, boxGrids3D.Ny, boxGrids3D.Nz

#x = (0:Nx3D-1) .* Δx .* 1e-3
#y = (0:Ny3D-1) .* Δy .* 1e-3 .- horizontalDepth*1.e-3
#z = (0:Nz3D-1) .* Δz .* 1e-3 .+ altMin*1.e-3

x = [p.xyz[1] for p in boxGrids3D.allGridsInCartesian[:,1,1]]*1.e-3
y = [p.xyz[2] for p in boxGrids3D.allGridsInCartesian[1,:,1]]*1.e-3
z = [p.xyz[3] for p in boxGrids3D.allGridsInCartesian[1,1,:]]*1.e-3
A = seismicModel3D.ρ

f = Figure()
ax = Axis3(f[1, 1])

volume!(ax,
    x[1] .. x[end],
    y[1] .. y[end],
    z[1] .. z[end],
    seismicModel3D.Vpv,
    algorithm = :absorption,   # optional, makes it nicer
    colormap = :viridis
)
f

In [ ]:
size(boxGrids3D.allGridsInCartesian)

In [ ]:
using CairoMakie

xvals = [p.xyz[1] for p in boxGrids3D.allGridsInCartesian[:,1,1]] .* 1e-3
zvals = [p.xyz[3] for p in boxGrids3D.allGridsInCartesian[1,1,590:621]] .* 1e-3

nx = length(xvals)
nz = length(zvals)

scale = 8

xmin, xmax = extrema(xvals)
zmin, zmax = extrema(zvals)

tmpCoordinates=[]
for i in 1:21

     
    yIndex = 10*(i-1)+1
    fig = Figure(resolution = (nx * scale, nz * scale))
    ax = Axis(fig[1, 1],
        aspect = DataAspect(),
        xlabel = "horizontal",
        ylabel = "depth from p1",
        title = "density model",
        xticks = xmin:20:xmax,
        yticks = zmin:10:zmax
    )

    tmpCoordinates=push!(tmpCoordinates,((boxGrids3D.allGridsInGeoPoints[1,yIndex,1].lat, boxGrids3D.allGridsInGeoPoints[1,yIndex,1].lon),
    (boxGrids3D.allGridsInGeoPoints[181,yIndex,1].lat, boxGrids3D.allGridsInGeoPoints[181,yIndex,1].lon)))
        

    hm = heatmap!(ax,
        xvals, zvals, seismicModel3D.ρ[:,yIndex,590:621];
        colormap = :seismic,
        interpolate = false
    )

    Colorbar(fig[1, 2], hm, label = "density")

    fig

    save("outputFuji_topo"*string(yIndex)*".png",fig)
end

In [ ]:
tmpCoordinates[11:21]

# another way of importing model parameters

## we can read images or cartoons

In [ ]:
modelName="marmousi"
imageFile="../dataInput/model/random/marmousi.png"
modelDefinitionMethod="2DimageFile" # ToyModel or 2DimageFile (or 1DsphericalPlanet)
model=defineModel(imageFile);

model construction 

In [ ]:
#
#boxGridsMarmousi = constructLocalBox(model,-3000.0,0.0,0.0,9200.0)
boxGridsMarmousi = lazyProduceOrLoad("MarmousiCoordInfo",constructLocalBox,model,-3000.0,0.0,0.0,9200.0)
#seismicModelMarmousi = makeAdHocSeismicModel(model, 1.0, 2.8, 1.5, 5.5, 0.0, 3.2)
seismicModelMarmousi=lazyProduceOrLoad("seismicModelMarmousi",makeAdHocSeismicModel,model, 1.0, 2.8, 1.5, 5.5, 0.0, 3.2)

#constructLocalBox for marmousi models should be written!

In [ ]:
using CairoMakie
xvals = [p.xz[1] for p in boxGridsMarmousi.allGridsInCartesian[:,1]]*1.e-3
zvals = [p.xz[2] for p in boxGridsMarmousi.allGridsInCartesian[1,:]]*1.e-3
fig, ax, hm = heatmap(
    #topo.x,topo.y,topo.z';
    #collect((0:1:(Nx-1)).*Δx).*1.e-3,(collect(0:1:(Nz-1)).*Δz.+altMin).*1.e-3, seismicModel.ρ;
    xvals, zvals, seismicModelMarmousi.Vsh;
    colormap = :seismic,
    #colorrange=(0,4),
    axis = (aspect = DataAspect(), xlabel = "horizontal", ylabel = "depth", title = "Vsh model")
)
Colorbar(fig[1,2], hm, label="Vsh")
fig

# Let's go to Mars

In [ ]:
set_default_planet!(:Mars)


# ok i need to change how to call 1D planet models too (because it is already called by DSM1D and it is not very much flexible but here I just use the same params as Earth)

In [ ]:
p1 = GeoPoint(15.0,135.0) # we should see Elysium planitia at least
p2 = GeoPoint(15.0,180.0) # 

In [ ]:
Δx = 3.e3 # in metre
Δy = 3.e3
Δz = 3.e3

horizontalDepthMin = -1000.e3
horizontalDepthMax = 1000.e3


altMax = 100.e3 # in metre
altMin = -600.e3 # in metre



In [ ]:
#allGridsInGeoPointsMars, allGridsInCartesianMars, effectiveRadiiMars=constructLocalBox(p1,p2,Δx,Δy,Δz,horizontalDepthMin,horizontalDepthMax,altMin,altMax)

In [ ]:
#NxM,NyM,NzM=size(allGridsInCartesianMars)

In [ ]:
#seismicModelMars=getParamsAndTopo(allGridsInGeoPointsMars,2.0)

In [ ]:
#using CairoMakie
#fig, ax, hm = heatmap(
#    collect((0:1:(Nx-1)).*Δx).*1.e-3,(collect(0:1:(Nz-1)).*Δz.+altMin).*1.e-3, seismicModelMars.Vsv[:,(Ny÷6)*1,:];
#    colormap = :seismic,
#    colorrange=(0,15),
#    axis = (aspect = DataAspect(), xlabel = "horizontal", ylabel = "depth from p1", title = "Vph model")
#)
#ylims!(ax,-200,300)
#xlims!(ax,0,400)
#Colorbar(fig[1,2], hm, label="P-wave")
#fig